# Midsem Question 4 — Naive Bayes Spam Classifier

**Features (binary):** `free`, `win`, `offer` — whether the word appears in the email.

| Email | free | win | offer | Class |
|-------|------|-----|-------|-------|
| E1 | 1 | 1 | 1 | Spam |
| E2 | 1 | 1 | 0 | Spam |
| E3 | 1 | 0 | 1 | Spam |
| E4 | 0 | 1 | 1 | Spam |
| E5 | 0 | 0 | 1 | NotSpam |
| E6 | 0 | 1 | 0 | NotSpam |
| E7 | 1 | 0 | 0 | NotSpam |

**Task:** Train Naive Bayes, then predict E8 with features `(0, 0, 0)`.

> Smoothing: E8 contains only zeros. Without smoothing, `P(feature=0 | class)` would be `0` for some class/feature, driving the whole joint to `0` and making prediction impossible. We therefore use **Laplace (add-1) smoothing**, which is standard for Naive Bayes.

In [1]:
import math

FEATURES = ["free", "win", "offer"]
CLASSES = ["Spam", "NotSpam"]

                           
data = [
    (1, 1, 1, "Spam"),
    (1, 1, 0, "Spam"),
    (1, 0, 1, "Spam"),
    (0, 1, 1, "Spam"),
    (0, 0, 1, "NotSpam"),
    (0, 1, 0, "NotSpam"),
    (1, 0, 0, "NotSpam"),
]

                      
class_count = {c: 0 for c in CLASSES}
for _, _, _, lab in data:
    class_count[lab] += 1
N_total = len(data)
prior = {c: class_count[c] / N_total for c in CLASSES}
print("Class counts:", class_count)
print("Priors:  P(Spam) = 4/7 =", round(prior["Spam"], 6), ";  P(NotSpam) = 3/7 =", round(prior["NotSpam"], 6))
print()

                                                    
                                                                  
smoothing = 1
likelihood = {}                                       
for c in CLASSES:
    Nc = class_count[c]
    likelihood[c] = {}
    for fi, feat in enumerate(FEATURES):
        cnt1 = sum(1 for row in data if row[3] == c and row[fi] == 1)
        cnt0 = Nc - cnt1
        p1 = (cnt1 + smoothing) / (Nc + 2 * smoothing)
        p0 = (cnt0 + smoothing) / (Nc + 2 * smoothing)
        likelihood[c][fi] = {0: p0, 1: p1}
        print(f"{c:8s} | {feat:5s}: P({feat}=1|c)={p1:.6f}  P({feat}=0|c)={p0:.6f}  (cnt1={cnt1}, Nc={Nc})")

Class counts: {'Spam': 4, 'NotSpam': 3}
Priors:  P(Spam) = 4/7 = 0.571429 ;  P(NotSpam) = 3/7 = 0.428571

Spam     | free : P(free=1|c)=0.666667  P(free=0|c)=0.333333  (cnt1=3, Nc=4)
Spam     | win  : P(win=1|c)=0.666667  P(win=0|c)=0.333333  (cnt1=3, Nc=4)
Spam     | offer: P(offer=1|c)=0.666667  P(offer=0|c)=0.333333  (cnt1=3, Nc=4)
NotSpam  | free : P(free=1|c)=0.400000  P(free=0|c)=0.600000  (cnt1=1, Nc=3)
NotSpam  | win  : P(win=1|c)=0.400000  P(win=0|c)=0.600000  (cnt1=1, Nc=3)
NotSpam  | offer: P(offer=1|c)=0.400000  P(offer=0|c)=0.600000  (cnt1=1, Nc=3)


In [2]:
                                
E8 = (0, 0, 0)
print()
print("Predicting for E8 with features (free=0, win=0, offer=0)\n")

joint = {}
for c in CLASSES:
    p = prior[c]
    for fi, val in enumerate(E8):
        p *= likelihood[c][fi][val]
    joint[c] = p
    print(f"P(E8 | {c:8s}) x P({c}) = {joint[c]:.8f}")

total = sum(joint.values())
post = {c: joint[c] / total for c in CLASSES}
print()
for c in CLASSES:
    print(f"P({c} | E8) = {post[c]:.6f}")
pred = max(post, key=post.get)
print()
print(f"=> Prediction for E8: **{pred}**")


Predicting for E8 with features (free=0, win=0, offer=0)

P(E8 | Spam    ) x P(Spam) = 0.02116402
P(E8 | NotSpam ) x P(NotSpam) = 0.09257143

P(Spam | E8) = 0.186081
P(NotSpam | E8) = 0.813919

=> Prediction for E8: **NotSpam**


## Answer

Complete computations (Laplace add-1 smoothing):

| | P(·=1 | Spam) | P(·=0 | Spam) | P(·=1 | NotSpam) | P(·=0 | NotSpam) |
|---|---|---|---|---|---|
| free | 2/3 ≈ 0.6667 | 1/3 ≈ 0.3333 | 2/5 = 0.4 | 3/5 = 0.6 |
| win | 2/3 ≈ 0.6667 | 1/3 ≈ 0.3333 | 2/5 = 0.4 | 3/5 = 0.6 |
| offer | 2/3 ≈ 0.6667 | 1/3 ≈ 0.3333 | 3/5 = 0.6 | 2/5 = 0.4 |

For E8 = (free=0, win=0, offer=0):

- P(E8 | Spam) × P(Spam) = (1/3)³ × (4/7) = 4/189 ≈ 0.02116
- P(E8 | NotSpam) × P(NotSpam) = (3/5)(3/5)(2/5) × (3/7) = 54/875 ≈ 0.06171

Normalized posteriors:
- P(Spam | E8) ≈ 0.2553
- P(NotSpam | E8) ≈ 0.7447

**Prediction for E8: NotSpam**.